# Introduction to the Rust (1987) Optimal Replacement Model

This notebook provides an introduction to the dynamic discrete choice model for optimal asset replacement, based on John Rust's seminal 1987 paper on bus engine replacement.

## Table of Contents

1. [Problem Setup](#problem-setup)
2. [The Decision Problem](#decision-problem)
3. [Dynamic Programming Formulation](#dynamic-programming)
4. [Simple Example](#simple-example)
5. [Key Insights](#insights)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

from src.cost_functions import LinearCost, QuadraticCost
from src.transitions import DiscreteTransition
from src.dp import DynamicProgram

## 1. Problem Setup <a name="problem-setup"></a>

Harold Zurcher manages a fleet of buses. Each month, he observes the mileage on each bus engine and must decide:

- **Keep**: Continue operating with the current engine, paying maintenance costs
- **Replace**: Install a new engine at a fixed cost, resetting mileage to zero

**Key Features:**
- Maintenance costs increase with mileage
- Replacement is costly but provides a "fresh start"
- Future is uncertain (stochastic mileage accumulation)
- Must trade off current costs vs. future benefits

## 2. The Decision Problem <a name="decision-problem"></a>

Let:
- $x_t$ = state (mileage) at time $t$
- $d_t \in \{0, 1\}$ = decision (0 = keep, 1 = replace)
- $c(x_t, \theta)$ = maintenance cost at state $x_t$
- $RC$ = replacement cost
- $\beta$ = discount factor

**Objective:** Maximize expected discounted utility

$$
\max E\left[\sum_{t=0}^{\infty} \beta^t u(x_t, d_t)\right]
$$

where utility is negative cost:

$$
u(x_t, d_t) = \begin{cases}
-c(x_t, \theta) & \text{if } d_t = 0 \text{ (keep)} \\
-RC & \text{if } d_t = 1 \text{ (replace)}
\end{cases}
$$

## 3. Dynamic Programming Formulation <a name="dynamic-programming"></a>

The value function $V(x)$ represents the maximum expected discounted utility starting from state $x$:

$$
V(x) = \max \left\{ V_0(x), V_1(x) \right\}
$$

where:

**Keep (d=0):**
$$
V_0(x) = -c(x, \theta) + \beta \int V(y) dF(y|x)
$$

**Replace (d=1):**
$$
V_1(x) = -RC + \beta \int V(y) dF(y|0)
$$

**Bellman Equation:**
$$
V(x) = \max\left\{ -c(x,\theta) + \beta \mathbb{E}[V(x')\mid x], \ -RC + \beta \mathbb{E}[V(x')\mid x'=0] \right\}
$$

## 4. Simple Example <a name="simple-example"></a>

Let's solve a simple example with:
- 50 mileage states (0-50 thousand miles)
- Linear maintenance cost: $c(x) = 0 + 0.002 \times x$
- Replacement cost: $RC = 5.0$ (thousand dollars)
- Discount factor: $\beta = 0.95$

In [ ]:
# Setup parameters
n_states = 50
discount_factor = 0.95
replacement_cost = 5.0

# Create cost function: c(x) = 0 + 0.002*x
cost_fn = LinearCost(theta=np.array([0.0, 0.002]))
state_grid = np.arange(n_states)
maintenance_costs = cost_fn.evaluate(state_grid)

# Create transition model (simple: 40% stay, 35% +1, 25% +2)
trans_model = DiscreteTransition(n_states, transition_probs=np.array([0.4, 0.35, 0.25]))
P = trans_model.get_transition_matrix()

print(f"Number of states: {n_states}")
print(f"Discount factor: {discount_factor}")
print(f"Replacement cost: ${replacement_cost}k")
print(f"Maintenance cost range: ${maintenance_costs[0]:.3f}k - ${maintenance_costs[-1]:.3f}k")

In [ ]:
# Visualize cost function
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Maintenance cost
axes[0].plot(state_grid, maintenance_costs, linewidth=2)
axes[0].axhline(replacement_cost, color='r', linestyle='--', label=f'Replacement cost = ${replacement_cost}k')
axes[0].set_xlabel('State (Mileage in 1000s)', fontsize=12)
axes[0].set_ylabel('Cost ($1000s)', fontsize=12)
axes[0].set_title('Maintenance vs. Replacement Cost', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Transition probabilities (from state 10)
axes[1].bar(state_grid, P[10, :], alpha=0.7)
axes[1].set_xlabel('Next State', fontsize=12)
axes[1].set_ylabel('Probability', fontsize=12)
axes[1].set_title('Transition Probabilities from State 10', fontsize=14)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Solve the dynamic program
dp = DynamicProgram(n_states, discount_factor, P, maintenance_costs, replacement_cost)
result = dp.value_iteration(verbose=True, max_iter=1000, tol=1e-6)

print(f"\nConverged: {result['converged']}")
print(f"Iterations: {result['iterations']}")
print(f"Time: {result['time']:.4f} seconds")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Value function
axes[0].plot(state_grid, result['value_function'], linewidth=2)
axes[0].set_xlabel('State (Mileage in 1000s)', fontsize=12)
axes[0].set_ylabel('Value Function', fontsize=12)
axes[0].set_title('Optimal Value Function V(x)', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Policy function
axes[1].step(state_grid, result['policy'], where='mid', linewidth=2)
axes[1].set_xlabel('State (Mileage in 1000s)', fontsize=12)
axes[1].set_ylabel('Decision (0=Keep, 1=Replace)', fontsize=12)
axes[1].set_title('Optimal Policy', fontsize=14)
axes[1].set_ylim(-0.1, 1.1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find replacement threshold
replace_states = np.where(result['policy'] == 1)[0]
if len(replace_states) > 0:
    threshold = replace_states[0]
    print(f"\nReplacement threshold: {threshold} thousand miles")
    print(f"Policy: Keep if mileage < {threshold}k, Replace if mileage ≥ {threshold}k")
else:
    print("\nNever replace (replacement cost too high)")

## 5. Key Insights <a name="insights"></a>

### Threshold Policy

The optimal policy typically has a **threshold structure**:
- Keep if $x < x^*$
- Replace if $x \geq x^*$

This makes intuitive sense: replace when mileage gets too high.

### Comparative Statics

The replacement threshold $x^*$ depends on:
- **Replacement cost (RC)**: Higher RC → higher threshold (keep longer)
- **Maintenance cost slope**: Steeper cost → lower threshold (replace sooner)
- **Discount factor (β)**: Higher β → more patient → may keep longer

### Value of Flexibility

The value function shows the **option value** of being able to choose optimally in the future.

### Next Steps

In the following notebooks, we'll explore:
- Simulation and data generation
- Parameter estimation using NFXP
- Applications to real-world scenarios